# 🎵 AICoverGen WebUI - Google Colab (O'zbekcha)
YouTube videolari yoki audio fayllardan istalgan RVC v2 ovoz modeli yordamida avtomatik AI cover (qo'shiq) yaratish.

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/nidayboy/aicovergenN/blob/main/AICoverGen_colab.ipynb)

---
### 📌 Qisqa ko'rsatma:
> 1. Yuqori menyudan **Runtime -> Change runtime type** bo'limida **T4 GPU** tanlanganligini tekshiring.
> 2. **1-QADAM:** Birinchi marta kirganda kutubxonalar va AI modellarni o'rnatish uchun ishga tushiring (~3-5 daqiqa).
> 3. **2-QADAM:** **Gradio** va/yoki **Ngrok** sozlamalarini belgilang, Ngrok tokeningizni kiriting va WebUI'ni ishga tushiring!
>    - *💡 Maslahat: Ngrok havolasi uzilib qolmaydi va juda tez ishlaydi. [ngrok.com](https://dashboard.ngrok.com/get-started/your-authtoken) saytidan bepul token olishingiz mumkin.*

In [ ]:
#@title 📦 1-QADAM: Kerakli kutubxona va modellarni o'rnatish (Faqat 1 marta ishga tushiring) { display-mode: "form" }
import os, sys, shutil, re
from IPython.display import clear_output

print("📦 1/5 Tizim paketlari o'rnatilmoqda (sox, ffmpeg)...")
!apt-get update -y -qq
!apt-get install -y -qq sox libsox-dev libsox-fmt-all ffmpeg

print("📦 2/5 Pip va mos kutubxonalar sozlanmoqda...")
!pip install -q --no-warn-conflicts "pip<24.1"
!pip install -q --no-warn-conflicts cython "numpy==1.26.4" hydra-core==1.0.7 omegaconf==2.0.6

print("📦 3/5 Fairseq kutubxonasi o'rnatilmoqda...")
if not os.path.exists("/content/fairseq"):
    !git clone https://github.com/pytorch/fairseq.git /content/fairseq

# Fairseq Python 3.13 OmegaConf muammosini to'liq bartaraf qilish
for fpath in ["/content/fairseq/fairseq/__init__.py", "/content/fairseq/fairseq/dataclass/initialize.py"]:
    if os.path.exists(fpath):
        with open(fpath, 'r', encoding='utf-8') as f: c = f.read()
        c = c.replace('hydra_init()', '# hydra_init()')
        c = re.sub(r'def hydra_init\([^)]*\)[^:]*:', 'def hydra_init(*args, **kwargs):\n    return', c)
        with open(fpath, 'w', encoding='utf-8') as f: f.write(c)

!pip install -q --no-warn-conflicts --no-build-isolation --editable /content/fairseq

print("📦 4/5 RVC audio, ONNX, pyngrok va Gradio kutubxonalari o'rnatilmoqda...")
!pip install -q --no-warn-conflicts pedalboard pydub pyworld praat-parselmouth faiss-cpu yt-dlp torchcrepe sox soundfile librosa deemix tqdm pyngrok onnxruntime-gpu gradio-client==0.8.1 "gradio==3.50.2" "fastapi<0.110" "starlette<0.37"

try:
    import gradio, urllib.request
    gradio_dir = os.path.dirname(gradio.__file__)
    frpc_path = os.path.join(gradio_dir, "frpc_linux_amd64_v0.2")
    if not os.path.exists(frpc_path):
        urllib.request.urlretrieve("https://cdn-media.huggingface.co/frpc-gradio-0.2/frpc_linux_amd64", frpc_path)
        os.chmod(frpc_path, 0o755)
except Exception:
    pass

print("📦 5/5 Repozitoriy va modellar tayyorlanmoqda...")
if not os.path.exists("/content/HRVC/src/webui.py"):
    if os.path.exists("/content/HRVC"):
        shutil.rmtree("/content/HRVC")
    !git clone https://github.com/nidayboy/aicovergenN.git /content/HRVC
else:
    %cd /content/HRVC
    !git fetch origin main && git reset --hard origin/main

%cd /content/HRVC
!python src/download_models.py

clear_output()
print("\n" + "="*60)
print("✅ BARCHA KUTUBXONA VA MODELLAR MUVAFFAQIYATLI O'RNATILDI!")
print("👉 Endi quyidagi 2-QADAM orqali WebUI'ni ishga tushiring.")
print("="*60 + "\n")


In [ ]:
#@title 🚀 2-QADAM: AICoverGen WebUI'ni ishga tushirish (Gradio va Ngrok) { display-mode: "form" }
#@markdown ---
#@markdown ### 🌐 1. Ulanish turini tanlang (Gradio va Ngrok):
Tunnel_turi = "Ikkalasi (Gradio + Ngrok)" #@param ["Ikkalasi (Gradio + Ngrok)", "Faqat Gradio", "Faqat Ngrok"]

#@markdown ### 🔑 2. Ngrok Token kiritish joyi (agar Ngrok tanlansa):
#@markdown > *Ngrok uzilib qolmaydigan tez va barqaror silka beradi. [Bu yerdan bepul token oling](https://dashboard.ngrok.com/get-started/your-authtoken)*
Ngrok_token = "" #@param {type:"string"}

Port = 7860 #@param {type:"integer"}

import os, sys, shutil, subprocess, re, time
from IPython.display import HTML, display

# Eski qolib ketgan jarayonlarni tozalash
!fuser -k 7860/tcp 2>/dev/null || true
!pkill -9 -f "webui.py" 2>/dev/null || true

# Fairseq Python 3.13 OmegaConf muammosini to'liq bartaraf qilish
for fpath in ["/content/fairseq/fairseq/__init__.py", "/content/fairseq/fairseq/dataclass/initialize.py"]:
    if os.path.exists(fpath):
        with open(fpath, 'r', encoding='utf-8') as f: c = f.read()
        c = c.replace('hydra_init()', '# hydra_init()')
        c = re.sub(r'def hydra_init\([^)]*\)[^:]*:', 'def hydra_init(*args, **kwargs):\n    return', c)
        with open(fpath, 'w', encoding='utf-8') as f: f.write(c)

# 1-qadam bajarilganligini tekshirish (agar bajarilmagan bo'lsa, avtomatik o'rnatish)
if not os.path.exists("/content/HRVC/src/webui.py"):
    print("⏳ Muhit hali o'rnatilmagan ekan. Avtomatik o'rnatish boshlanmoqda...")
    !apt-get update -y -qq
    !apt-get install -y -qq sox libsox-dev libsox-fmt-all ffmpeg
    !pip install -q --no-warn-conflicts "pip<24.1" cython "numpy==1.26.4" hydra-core==1.0.7 omegaconf==2.0.6 pyngrok
    if not os.path.exists("/content/fairseq"):
        !git clone https://github.com/pytorch/fairseq.git /content/fairseq
    for fpath in ["/content/fairseq/fairseq/__init__.py", "/content/fairseq/fairseq/dataclass/initialize.py"]:
        if os.path.exists(fpath):
            with open(fpath, 'r', encoding='utf-8') as f: c = f.read()
            c = c.replace('hydra_init()', '# hydra_init()')
            c = re.sub(r'def hydra_init\([^)]*\)[^:]*:', 'def hydra_init(*args, **kwargs):\n    return', c)
            with open(fpath, 'w', encoding='utf-8') as f: f.write(c)
    !pip install -q --no-warn-conflicts --no-build-isolation --editable /content/fairseq
    !pip install -q --no-warn-conflicts pedalboard pydub pyworld praat-parselmouth faiss-cpu yt-dlp torchcrepe sox soundfile librosa deemix tqdm onnxruntime-gpu gradio-client==0.8.1 "gradio==3.50.2" "fastapi<0.110" "starlette<0.37"
    if not os.path.exists("/content/HRVC/src/webui.py"):
        if os.path.exists("/content/HRVC"):
            shutil.rmtree("/content/HRVC")
        !git clone https://github.com/nidayboy/aicovergenN.git /content/HRVC
    %cd /content/HRVC
    !python src/download_models.py
else:
    # Repozitoriyni eng so'nggi yangilanishlarga sinxronlash
    %cd /content/HRVC
    !git fetch origin main && git reset --hard origin/main

%cd /content/HRVC

# Muhit parametrlarini sozlash
os.environ["TENSORBOARD_NO_TF"] = "1"

# pyngrok kutubxonasini tekshirish
!pip install -q --no-warn-conflicts pyngrok

# Gradio frpc binar faylini tekshirish
try:
    import gradio, urllib.request
    gradio_dir = os.path.dirname(gradio.__file__)
    frpc_path = os.path.join(gradio_dir, "frpc_linux_amd64_v0.2")
    if not os.path.exists(frpc_path):
        urllib.request.urlretrieve("https://cdn-media.huggingface.co/frpc-gradio-0.2/frpc_linux_amd64", frpc_path)
        os.chmod(frpc_path, 0o755)
except Exception:
    pass

use_gradio = Tunnel_turi in ["Ikkalasi (Gradio + Ngrok)", "Faqat Gradio"]
use_ngrok = Tunnel_turi in ["Ikkalasi (Gradio + Ngrok)", "Faqat Ngrok"]

# Ngrok tunnelni ishga tushirish
ngrok_url = None
if use_ngrok:
    token_toza = Ngrok_token.strip().replace('"', '').replace("'", "")
    if token_toza:
        try:
            from pyngrok import ngrok
            ngrok.set_auth_token(token_toza)
            ngrok.kill()
            tunnel = ngrok.connect(Port)
            ngrok_url = tunnel.public_url
            print(f"🚀 NGROK SILKASI: {ngrok_url}")
        except Exception as e:
            print(f"⚠️ Ngrok xatosi: {e}")
    else:
        print("⚠️ DIQQAT: Ngrok tanlangan, lekin 'Ngrok_token' kiritilmagan!")
        print("💡 Ngrok ishlashi uchun https://dashboard.ngrok.com dan bepul token oling.")
        if not use_gradio:
            print("ℹ️ Ulanish imkoni bo'lishi uchun Gradio avtomatik yoqildi!")
            use_gradio = True

cmd = ["python", "-u", "src/webui.py", "--listen", "--listen-port", str(Port)]
if use_gradio:
    cmd.append("--share")

print("\n" + "="*60)
print("⏳ AICoverGen WebUI ishga tushirilmoqda, iltimos kuting...")
print("="*60 + "\n")

process = subprocess.Popen(
    cmd,
    stdout=subprocess.PIPE,
    stderr=subprocess.STDOUT,
    text=True,
    bufsize=1,
    universal_newlines=True
)

gradio_url = None
banner_shown = False

def show_links_banner(g_url, n_url):
    buttons_html = ""
    if g_url:
        buttons_html += f'''
        <a href="{g_url}" target="_blank" style="background-color: #ff5722; color: white; padding: 12px 24px; text-decoration: none; border-radius: 8px; font-weight: bold; font-size: 16px; margin: 8px; display: inline-block; box-shadow: 0 4px 10px rgba(255,87,34,0.4);">
            🌐 GRADIO KIRISH SILKASI
        </a>
        '''
    if n_url:
        buttons_html += f'''
        <a href="{n_url}" target="_blank" style="background-color: #2196f3; color: white; padding: 12px 24px; text-decoration: none; border-radius: 8px; font-weight: bold; font-size: 16px; margin: 8px; display: inline-block; box-shadow: 0 4px 10px rgba(33,150,243,0.4);">
            🚀 NGROK KIRISH SILKASI
        </a>
        '''
    
    html = f'''
    <div style="background: #181824; border: 2px solid #4f46e5; border-radius: 12px; padding: 20px; text-align: center; font-family: -apple-system, BlinkMacSystemFont, 'Segoe UI', Roboto, sans-serif; margin: 15px 0;">
        <h2 style="color: #ffffff; margin-top: 0; font-size: 22px;">🎉 AICoverGen WebUI Tayyor!</h2>
        <p style="color: #9ca3af; font-size: 14px; margin-bottom: 15px;">Dasturdan foydalanish uchun quyidagi silka (havola) ustiga bosing:</p>
        <div>
            {buttons_html}
        </div>
        <div style="margin-top: 15px; color: #d1d5db; font-size: 14px;">
            {f'<p>🔗 <b>Gradio:</b> <a href="{g_url}" target="_blank" style="color: #ff7043;">{g_url}</a></p>' if g_url else ''}
            {f'<p>🔗 <b>Ngrok:</b> <a href="{n_url}" target="_blank" style="color: #64b5f6;">{n_url}</a></p>' if n_url else ''}
        </div>
    </div>
    '''
    display(HTML(html))

# Agar faqat Ngrok bo'lsa (Gradio o'chiq bo'lsa), darhol bannerni ko'rsatish
if not use_gradio and ngrok_url:
    show_links_banner(None, ngrok_url)
    banner_shown = True

for line in iter(process.stdout.readline, ''):
    print(line, end='', flush=True)
    if ("gradio.live" in line or "gradio.app" in line) and not banner_shown:
        match = re.search(r'(https://[a-zA-Z0-9-]+\.(?:gradio\.live|gradio\.app))', line)
        if match:
            gradio_url = match.group(1)
            show_links_banner(gradio_url, ngrok_url)
            banner_shown = True

process.wait()
